# Depolarizing noise on a 1D chain: $\kappa$, $\kappa_0$ and $\lambda_{\rm eff}$

This notebook does not benchmark a device. It justifies the model used to turn a device's LR-QAOA
results into one effective error rate per edge, $\lambda_{\rm eff}$, and it computes the constants that
model needs.

**The question.** Run LR-QAOA of depth $p$ on an $N_q$-spin Ising chain $H=\sum_{(i,j)\in E} Z_iZ_j$, whose
$N_{\rm edges} = N_q - 1$ edges are the bonds $(i, i+1)$. Every layer applies each edge's interaction
$e^{-i\gamma Z_iZ_j}$ once. Put a two-qubit depolarizing channel of strength $\lambda$ on the two qubits
after every edge interaction,

$$\mathcal{E}(\rho) = (1-\lambda)\,\rho + \lambda\,\frac{I}{4},$$

and nothing else. How much of the noiseless answer survives?

**Why per edge, not per gate.** How an edge interaction is built depends on the implementation. The
*direct* circuit spends two CNOTs on it (`CX RZ CX`). The *MCM* circuit spends CZs to an ancilla, a
mid-circuit measurement and feed-forward. Counting errors per edge leaves those resources out of the
model: $\lambda$ is whatever one edge interaction costs, however it was made. The direct and MCM circuits
of the same Hamiltonian are then measured in the same unit, and the difference between their
$\lambda_{\rm eff}$ is what the MCM resources cost relative to the CNOTs.

**The metrics.** For the approximation ratio $r$ and for the probability $\rho$ of sampling an optimum,
the *overlap* measures how far the noisy result has fallen from the noiseless value toward random
guessing:

$$r_{\rm ovl} = \frac{r - r_{\rm rand}}{r_{\rm ideal} - r_{\rm rand}}, \qquad
  \rho_{\rm ovl} = \frac{\rho - \rho_{\rm rand}}{\rho_{\rm ideal} - \rho_{\rm rand}},$$

where $r_{\rm rand} = 1/2$ and $\rho_{\rm rand} = 2/2^{N_q}$ (the chain has two optimal states). An
overlap of 1 is noiseless and 0 is random.

**The model.** The circuit contains $N_{\rm edges}\,p$ noisy edge interactions, so the *accumulated
error* is

$$\varepsilon_{\rm acc} = N_{\rm edges}\;p\;\lambda .$$

The hypothesis is that every $(N_q, p, \lambda)$ falls on one exponential

$$r_{\rm ovl},\ \rho_{\rm ovl} = 2^{-\kappa\,\varepsilon_{\rm acc}},$$

with $\kappa_r = \kappa_0 / N_q$ for the approximation ratio: each fault destroys a fixed fraction of the
$N_{\rm edges}$ edges' worth of signal. For the probability of the optimum, $\kappa_\rho$ is close to 1 at every
size, because one fault is enough to leave the optimum.

**What it is used for.** With $\kappa$ fixed, a device's overlaps at several depths determine a single
unknown, $\lambda_{\rm eff}$: the depolarizing strength per edge interaction that would degrade the circuit as
much as the device did. Section 6 does this for the direct and mid-circuit-measurement runs of a 10-spin chain on
`ibm_boston` and on `Helios-1E`, Quantinuum's emulator of Helios-1. Section 7 extends it to chains of
10 to 60 spins on eight devices, direct and MCM.

## How the simulation stays fast

A density-matrix simulation at 10 000 shots for every $(N_q, p, \lambda)$ takes hours at 15 qubits.
Two exact identities avoid density matrices altogether. The method is exact in expectation and is
checked against a density matrix in section 3.

1. **A fault only flips the sign of some angles.** The channel does nothing with probability $1-q$,
   $q = \tfrac{15}{16}\lambda$, and otherwise applies one of the 15 non-identity two-qubit Paulis.
   Carried forward through the circuit, an $X$ on one qubit of a later $R_{ZZ}$ reverses that rotation,
   a $Z$ reverses a later $R_X$, and whatever reaches the end only relabels the measured bits. Each
   noisy run is an ordinary pure-state LR-QAOA with some angles negated.
2. **One simulation covers every $\lambda$.** Given that exactly $k$ of the $N_{\rm edges}\,p$ edge
   interactions failed, which ones failed, and how, does not depend on $\lambda$. So
   $$\langle O\rangle(\lambda) = \sum_k \mathrm{Binomial}\big(k;\ N_{\rm edges}\,p,\ q(\lambda)\big)\ m_k ,$$
   and the conditional means $m_k$ are estimated once per $(N_q, p)$. Every $\lambda$ afterwards is a
   weighted sum.

$m_k$ is computed exactly (every configuration) when that is cheap, and otherwise averaged over sampled
configurations, with its standard error carried into every overlap. Sampling stops once $m_k$ is
indistinguishable from random.

The overlaps hardly depend on which depths are used, so a few depths per size are enough.

In [ ]:
import json
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))          # not needed after `pip install -e .`
DATA = ROOT / "data" / "noise_study"
RESULTS = ROOT / "data" / "results"
FIGURES = ROOT / "figures" / "noise_study"
FIGURES.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np

from qecbench.analysis import load_results
from qecbench.noise_study import (accumulated_error, collapse, device_overlaps, fit_kappa, fit_kappa0,
                                  decaying_part, device_run, fit_lambda_eff, lambda_from_cx, overlap_model,
                                  run_study, simulate_decay)

## 1. Configuration

* `NQS`, `DEPTHS` - chain lengths and LR-QAOA depths to simulate. The cost grows as $2^{N_q}$ and
  linearly in $p$. On a laptop, $N_q = 5\ldots15$ at $p = 2, 5, 10$ takes a few minutes.
* `LAMBDAS` - depolarizing strengths per edge interaction at which overlaps are evaluated. They cost
  nothing extra (see above).
* `TRAJECTORIES` - sampled configurations per number of faults $k$. Error bars scale as
  $1/\sqrt{\texttt{TRAJECTORIES}}$.
* `RECOMPUTE` - results are cached in `data/noise_study/`. Set `True` to discard the cache.
* `DEVICE_RUNS` - the device runs of section 6: backend, kind and run files in
  `data/results/<backend>/chain/<kind>/`, and the depths used. `DEVICE_SIM_DEPTHS` - extra depths simulated for that chain length only, so that
  $\kappa_\rho$ is fitted at the depths the devices ran (see section 5).
* `LAMBDA_SERIES`, `LAMBDA_WINDOWS`, `DROP_PLATEAU` - section 7: for each device and circuit, the run files
  (`backend`, `stamp`) read from `data/results/<backend>/chain/<kind>/`, and how their depths are fitted.

In [ ]:
NQS          = list(range(5, 16))
DEPTHS       = [2, 5, 10]
DELTA        = 0.5
LAMBDAS      = np.logspace(-5, 0, 25)
TRAJECTORIES = 128
SEED         = 2026
RECOMPUTE    = False
CACHE        = DATA / "chain_depolarizing.json"

# device runs of a 10-spin chain: label -> (backend, kind, run stamps in data/results, depths used)
DEVICE_N    = 10
DEVICE_SIM_DEPTHS = [10, 20, 30, 40, 50]   # simulated at N_q = DEVICE_N only, to match the device depths
DEVICE_RUNS = {
    ("ibm_boston", "direct"): ("ibm_boston", "direct", ["20260506_1507"], [10, 15, 20, 30, 40, 50]),
    ("ibm_boston", "MCM"):    ("ibm_boston", "mcm",    ["20260506_1537"], [10, 15, 20, 30, 40, 50]),
    ("Helios-1E", "direct"):  ("Helios-1E",  "direct", ["20260507_0900"], [*range(1, 11), 15, 20, 30, 40, 50]),
    ("Helios-1E", "MCM"):     ("Helios-1E",  "mcm",    ["20260508_0700"], [*range(1, 11), 15, 20, 30, 40, 50]),
}

# lambda_eff against chain length (section 7): device -> {circuit: [(backend, run stamp), ...]}
LAMBDA_SERIES = {
    "ibm_boston":        {"direct": [("ibm_boston", s) for s in ("20260512_1429", "20260428_110845", "20260428_110857",
                                                                  "20260428_110905", "20260428_110915", "20260430_0834")],
                          "mcm":    [("ibm_boston", "20260827_1035")]},
    "ibm_pittsburgh":    {"direct": [("ibm_pittsburgh", s) for s in ("20260512_1402", "20260428_111001", "20260428_110953",
                                                                      "20260428_110944", "20260428_110932", "20260430_1618")],
                          "mcm":    [("ibm_pittsburgh", "20260828_1025")]},
    "ibm_kingston":      {"direct": [("ibm_kingston", s) for s in ("20260512_1402", "20260505_0806", "20260505_0805")],
                          "mcm":    [("ibm_kingston", "20260827_1205")]},
    "quantinuum_helios": {"direct": [("Helios-1", s) for s in ("20260525_0957", "20260309_0726", "20260526_0802")],
                          "mcm":    [("Helios-1", s) for s in ("20260915_1035", "20260909_1423", "20260909_1509", "20260909_1521")]},
    "quantinuum_h2":     {"direct": [("H2-1E", "20260820_0646"), ("H2-1", "20260820_0651")],
                          "mcm":    [("H2-1E", "20260820_0647"), ("H2-1", "20260820_0650")]},
    "ibm_fez":           {"mcm": [("ibm_fez", "20260828_1028")]},
    "ibm_marrakesh":     {"mcm": [("ibm_marrakesh", "20260828_1034")]},
    "ibm_phoenix":       {"mcm": [("ibm_phoenix", "20260904_1435")]},
}
LAMBDA_COLOURS = {"ibm_boston": 0, "ibm_pittsburgh": 1, "ibm_kingston": 2, "quantinuum_helios": 3,
                  "quantinuum_h2": 4, "ibm_fez": 6, "ibm_marrakesh": 7, "ibm_phoenix": 8}   # Set1 indices
QUANTINUUM_HARDWARE = {"Helios-1", "H2-1"}                                                  # drawn dotted
LAMBDA_WINDOWS = {"direct": [3, 5, 10, 15, 20, 30, 40, 50], "mcm": [3, 5, 10]}
DROP_PLATEAU   = {"direct": True, "mcm": False}

## 2. Simulate

For every $(N_q, p)$ this prints the largest number of faults $k$ simulated, out of the $N_{\rm edges}\,p$
edge interactions, the number of pure-state runs, and the time taken. Cached tables print nothing.

In [ ]:
t0 = time.perf_counter()
decays = run_study(NQS, DEPTHS, delta=DELTA, trajectories=TRAJECTORIES, seed=SEED,
                   cache=CACHE, recompute=RECOMPUTE)
print(f"{len(decays)} tables ready in {time.perf_counter() - t0:.1f} s  ->  {CACHE.relative_to(ROOT)}")

## 3. Check: identical to a density matrix

On a chain small enough to enumerate every fault configuration, the method above is compared with Aer's
density-matrix simulation: an `RZZ` gate for every edge, followed by the depolarizing channel. They
should agree to rounding error, which confirms the sign flips and the binomial sum.

The last column connects this to a gate-level model: the same edges written as `CX RZ CX` with a
channel of strength $\lambda_{\rm CX}$ after *each* CNOT. A depolarizing channel commutes with any gate on
its two qubits, so that is exactly the per-edge model at $\lambda = 1-(1-\lambda_{\rm CX})^2 \approx
2\lambda_{\rm CX}$: the per-edge $\lambda$ of a direct circuit counts both of its CNOTs.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error


def density_matrix_r(n, depth, lam, delta=DELTA, per="edge"):
    gammas = [(k + 1) * delta / depth for k in range(depth)]
    betas = [(depth - k) * delta / depth for k in range(depth)]
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for gamma, beta in zip(gammas, betas):
        for i in range(n - 1):
            if per == "edge":
                qc.rzz(2 * gamma, i, i + 1)
            else:
                qc.cx(i, i + 1); qc.rz(2 * gamma, i + 1); qc.cx(i, i + 1)
        qc.rx(-2 * beta, range(n))
    zz = SparsePauliOp.from_sparse_list([("ZZ", [i, i + 1], 1.0) for i in range(n - 1)], num_qubits=n)
    qc.save_expectation_value(zz, range(n))
    noise = NoiseModel()
    noise.add_all_qubit_quantum_error(depolarizing_error(lam, 2), ["rzz" if per == "edge" else "cx"])
    energy = AerSimulator(method="density_matrix", noise_model=noise).run(qc).result().data(0)["expectation_value"]
    return ((n - 1) - energy) / (2 * (n - 1))


def r_at(decay, lam):
    return 0.5 + decay.overlap(lam, "r")[0][0] * (decay.r_ideal - 0.5)


check = simulate_decay(4, 1, delta=DELTA, exhaustive_limit=10 ** 6)
assert all(check.exact)
print(f"N_q = 4, p = 1: all {sum(check.runs)} fault configurations (every k up to {check.slots}) enumerated\n")
print(f"{'lambda':>8} {'r (this method)':>16} {'density matrix':>15}   |  {'lambda_CX':>9} {'CX-level density matrix':>24}")
for lam in (1e-3, 1e-2, 1e-1, 1.0):
    lam_cx = 1 - np.sqrt(1 - lam)                      # the lambda_CX with lambda_from_cx(lambda_CX) = lambda
    assert np.isclose(lambda_from_cx(lam_cx), lam)
    print(f"{lam:8.0e} {r_at(check, lam):16.12f} {density_matrix_r(4, 1, lam):15.12f}   |  "
          f"{lam_cx:9.2e} {density_matrix_r(4, 1, lam_cx, per='cx'):24.12f}")

## 4. $\kappa_r$ for each chain length, and $\kappa_0$

Left: overlaps of the approximation ratio against $\varepsilon_{\rm acc}$, one colour per $N_q$ and all
depths together. Every depth of a given size lands on the same curve. Lines are the fits
$2^{-\kappa_r \varepsilon_{\rm acc}}$, one per size.

Right: the fitted $\kappa_r$ against $N_q$ (error bars are $3\sigma$) with the one-parameter law
$\kappa_r = \kappa_0/N_q$.

Fits use every point with a positive overlap, as in the density-matrix study. Points far into the random
regime contribute little either way.

In [ ]:
fits_r = {}
for n in NQS:
    eps, ovl, err, depth, tail = collapse(decays, n, LAMBDAS, "r")
    fits_r[n] = fit_kappa(eps, ovl)
kappa0, kappa0_err, kappa0_rss = fit_kappa0(NQS, [fits_r[n]["kappa"] for n in NQS])

print(" N_q   kappa_r            N_q * kappa_r   R^2")
for n in NQS:
    f = fits_r[n]
    print(f"{n:4d}   {f['kappa']:.4f} ± {f['stderr']:.4f}   {n * f['kappa']:.3f}          {f['r2']:.5f}")
print(f"\nkappa_0 = {kappa0:.3f} ± {kappa0_err:.3f}   (kappa_r = kappa_0 / N_q, RSS = {kappa0_rss:.2e})")

cmap = plt.get_cmap("jet")
colour = {n: cmap(i / max(len(NQS) - 1, 1)) for i, n in enumerate(NQS)}
fig, (ax, bx) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.6, 1]})
x = np.logspace(-2, 2, 300)
for n in NQS:
    eps, ovl, err, depth, tail = collapse(decays, n, LAMBDAS, "r")
    keep = (eps >= 1e-2) & (eps <= 1e2)
    ax.errorbar(eps[keep], ovl[keep], yerr=err[keep], fmt="o", ms=3.5, color=colour[n], alpha=0.75,
                label=f"$N_q$={n}" if n % 2 else None)
    ax.plot(x, overlap_model(x, fits_r[n]["kappa"]), color=colour[n], lw=1)
ax.set(xscale="log", xlim=(1e-2, 1e2), ylim=(-0.05, 1.05), xlabel=r"$\varepsilon_{acc} = N_{edges}\,p\,\lambda$",
       ylabel=r"$r_{\rm ovl}$", title="approximation-ratio overlap, all depths")
ax.legend(fontsize=8, ncol=2)

ks = np.array([fits_r[n]["kappa"] for n in NQS])
es = np.array([fits_r[n]["stderr"] for n in NQS])
for n, k, e in zip(NQS, ks, es):
    bx.errorbar(n, k, yerr=3 * e, fmt="o", color=colour[n], capsize=3)
dense = np.linspace(min(NQS) - 0.5, max(NQS) + 0.5, 200)
bx.plot(dense, kappa0 / dense, ":", color="purple", lw=2, label=fr"$\kappa_0/N_q$, $\kappa_0$={kappa0:.2f}")
bx.set(yscale="log", xlabel="$N_q$", ylabel=r"$\kappa_r$", title=r"$\kappa_r$ against chain length")
bx.legend()
fig.tight_layout()
fig.savefig(FIGURES / "kappa_r_scaling.pdf", bbox_inches="tight")
plt.show()

## 5. $\kappa_\rho$: the probability of the optimum

Same analysis for $\rho_{\rm ovl}$. A single fault almost always moves the output away from the two
optimal states, so $\kappa_\rho$ stays near 1 whatever the size, unlike $\kappa_r$. Error bars grow with
$N_q$ because $\rho_{\rm ideal}$ itself shrinks.

Unlike $\kappa_r$, $\kappa_\rho$ also drifts slowly with depth: at $N_q = 10$ it is about 0.94 over
$p = 2\ldots10$ and about 1.0 over $p = 15\ldots50$. So section 6 fits it again at the depths the devices
actually ran.

In [ ]:
fits_rho = {}
for n in NQS:
    eps, ovl, err, depth, tail = collapse(decays, n, LAMBDAS, "prob")
    fits_rho[n] = fit_kappa(eps, ovl)
print(" N_q   kappa_rho          R^2")
for n in NQS:
    f = fits_rho[n]
    print(f"{n:4d}   {f['kappa']:.4f} ± {f['stderr']:.4f}   {f['r2']:.5f}")

fig, (ax, bx) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.6, 1]})
for n in NQS:
    eps, ovl, err, depth, tail = collapse(decays, n, LAMBDAS, "prob")
    keep = (eps >= 1e-2) & (eps <= 1e2)
    ax.errorbar(eps[keep], ovl[keep], yerr=err[keep], fmt="o", ms=3.5, color=colour[n], alpha=0.75,
                label=f"$N_q$={n}" if n % 2 else None)
    ax.plot(x, overlap_model(x, fits_rho[n]["kappa"]), color=colour[n], lw=1)
ax.set(xscale="log", xlim=(1e-2, 1e2), ylim=(-0.05, 1.05), xlabel=r"$\varepsilon_{acc}$", ylabel=r"$\rho_{\rm ovl}$",
       title="probability-of-optimum overlap, all depths")
ax.legend(fontsize=8, ncol=2)
for n in NQS:
    bx.errorbar(n, fits_rho[n]["kappa"], yerr=3 * fits_rho[n]["stderr"], fmt="o", color=colour[n], capsize=3)
bx.set(xlabel="$N_q$", ylabel=r"$\kappa_\rho$", title=r"$\kappa_\rho$ against chain length")
fig.tight_layout()
fig.savefig(FIGURES / "kappa_rho_scaling.pdf", bbox_inches="tight")
plt.show()

## 6. $\lambda_{\rm eff}$ of real devices ($N_q = 10$)

The device data are four runs of the 10-spin chain, in this repository's result format under
`data/results/<backend>/chain/<kind>/`: `ibm_boston` hardware at 10 000 shots per circuit (6 May 2026) and
`Helios-1E`, Quantinuum's emulator of Helios-1 with its noise model, at 1 000 shots (7-8 May 2026, flagged as
simulated). Each device ran the circuit both directly (`direct`) and with mid-circuit measurements (`mcm`).
They were converted from earlier result files (`<stamp>_<backend>_1d_<normal|MCM>_nq10_depth<p>.json`)
by `scripts/import_legacy_chains.py`, so $r$ and the probability of the optimum come from the samples. The
run files hold every depth that was run. `DEVICE_RUNS` selects the depths used here, the same as in the
published analysis: $p = 10\ldots50$ for `ibm_boston` and $p = 1\ldots10, 15, 20, 30, 40, 50$ for `Helios-1E`.

For each device run, the overlaps at every depth are computed against the exact noiseless and random
references. Then $\lambda_{\rm eff}$ is fitted with $\kappa$ held fixed:

* approximation ratio: $\kappa_r = \kappa_0 / N_q$, from section 4;
* probability of the optimum: $\kappa_\rho$ fitted at $N_q = 10$ over `DEVICE_SIM_DEPTHS` (the deep
  runs the devices made), because $\kappa_\rho$ depends on depth.

Each device point is placed at $\varepsilon_{\rm acc} = N_{\rm edges}\,p\,\lambda_{\rm eff}$ with
$N_{\rm edges} = 9$. A device whose points follow the simulated curve (grey) degrades like uniform
depolarizing noise, and $\lambda_{\rm eff}$ is then a fair one-number summary.

$\lambda_{\rm eff}$ is an error **per edge interaction**. For `direct` it covers the two CNOTs of each
edge. For `MCM` it covers the CZs to the ancilla, the ancilla measurement and the feed-forward. The ratio
$\lambda_{\rm eff}^{\rm MCM} / \lambda_{\rm eff}^{\rm direct}$ printed below is therefore how much more one
edge costs through mid-circuit measurement than through CNOTs, on the same device. Overlaps above 1.2, which are shot noise on nearly
noiseless points, are set to 1, as in the original analysis.

In [ ]:
deep = run_study([DEVICE_N], DEVICE_SIM_DEPTHS, delta=DELTA, trajectories=TRAJECTORIES, seed=SEED,
                 cache=CACHE, recompute=RECOMPUTE)
deep_r = fit_kappa(*collapse(deep, DEVICE_N, LAMBDAS, "r")[:2])
deep_rho = fit_kappa(*collapse(deep, DEVICE_N, LAMBDAS, "prob")[:2])
print(f"N_q = {DEVICE_N}, p = {DEVICE_SIM_DEPTHS}:  kappa_r = {deep_r['kappa']:.4f} "
      f"(kappa_0/N_q = {kappa0 / DEVICE_N:.4f})   kappa_rho = {deep_rho['kappa']:.4f} ± {deep_rho['stderr']:.4f}\n")
kappas = {"r": kappa0 / DEVICE_N, "prob": deep_rho["kappa"]}
names = {"r": r"$r_{\rm ovl}$", "prob": r"$\rho_{\rm ovl}$"}
device_colour = {"ibm_boston": plt.get_cmap("Set1")(0), "Helios-1E": plt.get_cmap("Set1")(1)}
marker = {"direct": "o", "MCM": "^"}
lambda_eff = {}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, observable in zip(axes, ("r", "prob")):
    eps, ovl, err, depth, tail = collapse(deep, DEVICE_N, LAMBDAS, observable)
    ax.plot(eps, ovl, "o", color="gray", alpha=0.4, ms=4, label="depolarizing simulation")
    ax.plot(x, overlap_model(x, kappas[observable]), "--", color="black", lw=2,
            label=fr"$2^{{-\kappa\,\varepsilon_{{acc}}}}$, $\kappa$={kappas[observable]:.3f}")
    for (device, circuit), (backend, kind, stamps, depths) in DEVICE_RUNS.items():
        run = device_run(RESULTS, backend, kind, stamps, DEVICE_N, depths)
        if not run:
            print(f"no {kind} run of N_q = {DEVICE_N} for {device} in {stamps}")
            continue
        ps, values = device_overlaps(run, DEVICE_N, DELTA, observable)
        fit = fit_lambda_eff(ps, values, DEVICE_N - 1, kappas[observable])
        lambda_eff[(device, circuit, observable)] = fit
        ax.plot(accumulated_error(DEVICE_N - 1, ps, fit["lambda_eff"]), values, marker[circuit], ms=8,
                color=device_colour[device], mec="black", label=f"{device} {circuit}")
    ax.set(xscale="log", xlim=(1e-2, 1e2), ylim=(-0.05, 1.1), xlabel=r"$\varepsilon_{acc}$",
           ylabel=names[observable], title=f"$N_q$ = {DEVICE_N}")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / f"lambda_eff_nq{DEVICE_N}.pdf", bbox_inches="tight")
plt.show()

print("lambda_eff per edge interaction")
print("                          from r                         from rho")
for device, circuit in DEVICE_RUNS:
    cells = []
    for observable in ("r", "prob"):
        f = lambda_eff.get((device, circuit, observable))
        cells.append(f"{f['lambda_eff']:.2e} ± {f['stderr']:.1e} (rmse {f['rmse']:.3f})" if f else "-")
    print(f"{device:>11s} {circuit:<7s}   " + "   ".join(cells))

print("\nMCM / direct, per edge")
for device in dict.fromkeys(d for d, _ in DEVICE_RUNS):
    ratios = [lambda_eff[(device, "MCM", o)]["lambda_eff"] / lambda_eff[(device, "direct", o)]["lambda_eff"]
              for o in ("r", "prob") if (device, "MCM", o) in lambda_eff and (device, "direct", o) in lambda_eff]
    print(f"{device:>11s}   " + "   ".join(f"{name}: {x:.2f}" for name, x in zip(("from r", "from rho"), ratios)))

## 7. $\lambda_{\rm eff}$ against chain length, across devices

The same per-edge fit, at every chain length a device ran:

$$r_{\rm ovl}(p) = 2^{-(\kappa_0/N_q)\,(N_q-1)\,p\,\lambda_{\rm eff}} ,$$

with $\kappa_0$ from section 4. Chains here run to $N_q = 60$, far past the $N_q \le 15$ that $\kappa_0$ was
fitted on, so the $1/N_q$ law is extrapolated. That is the one assumption this section adds.

**Data.** All of it is in this repository, converted into its result format under
`data/results/<backend>/chain/<direct|mcm>/` by `scripts/import_legacy_chains.py`. Each file records its
source. $r$, its shot-noise error, $r_{\rm rand} = 1/2$ and $r_{\rm ideal}$ are recomputed from the stored
samples. $r_{\rm ideal}$ is exact at every length: an open chain is a free-fermion problem, so no
simulator shot noise or truncation enters. Devices that reuse ancillas (Quantinuum) get logical ancilla
labels, flagged in each file.

| device | direct circuits (circles, solid) | MCM circuits (crosses, dashed) |
|---|---|---|
| `ibm_boston`, `ibm_pittsburgh`, `ibm_kingston` | April-May 2026, $N_q = 10\ldots60$ | late August 2026, $N_q = 10\ldots60$ |
| Quantinuum Helios | Helios-1 hardware, $N_q = 30, 40, 50$ | Helios-1 hardware, $N_q = 20\ldots50$ (September 2026) |
| Quantinuum H2 | H2-1E emulator $N_q = 10, 20$; H2-1 hardware $N_q = 30$ | same split, 20 August 2026 |
| `ibm_fez`, `ibm_marrakesh`, `ibm_phoenix` | - | late August / early September 2026 |

Dotted segments are Quantinuum **hardware**, and a dotted link joins an emulator point to a hardware
point. Helios-1E emulator runs are left out, so that no simulated point looks measured.

**Read the MCM curves against each other, not against the direct ones.** The MCM campaigns use the
two-colour ordering of the gadgets, a constant circuit depth of 2 per layer. The earlier direct campaigns
come from a different period, and no direct runs were made alongside the new MCM ones.

**Fit windows.**
* MCM: $p \in \{3, 5, 10\}$, the only depths every device ran (Helios-1 stops at $p = 10$). The early
  points sit above a pure exponential, so this window inflates $\lambda_{\rm eff}$ by roughly 1.2-1.5x
  compared with a $p \ge 10$ window, but it does so for every device alike.
* direct: every depth run among $3\ldots50$, after dropping the leading plateau (points up to the
  maximum overlap), where shot noise can make the overlap rise with depth.

Points at or below the random floor carry no information and are dropped. There are no error bars: with
2-8 points per fit, the curve-fit covariance mostly reflects how many points survived, not how well they
were measured. The fit RMSE is in the table.

In [ ]:
def fit_runs(backend, kind, stamps):
    files = {f"{stamp}_{backend}_chain_{kind}.json" for stamp in stamps}
    fits = {}
    for chain, by_depth in load_results(RESULTS, backend, kind=kind, files=files).items():
        n = chain.n_data
        ps = np.array([p for p in LAMBDA_WINDOWS[kind] if p in by_depth], dtype=float)
        ovl = np.array([by_depth[int(p)]["r_ovl"] for p in ps])
        ps, ovl = ps[ovl > 0], ovl[ovl > 0]
        if DROP_PLATEAU[kind]:
            ps, ovl = decaying_part(ps, ovl)
        if len(ps) < 2:
            continue
        fits[n] = {**fit_lambda_eff(ps, ovl, n - 1, kappa0 / n), "depths": ps.tolist(), "backend": backend,
                   "shots": by_depth[int(ps[0])]["shots"]}
    return fits


t0 = time.perf_counter()
lambda_by_nq = {}                                   # (device, kind) -> {N_q: fit}
for device, kinds in LAMBDA_SERIES.items():
    for kind, runs in kinds.items():
        by_backend = {}
        for backend, stamp in runs:
            by_backend.setdefault(backend, []).append(stamp)
        lambda_by_nq[(device, kind)] = {}
        for backend, stamps in by_backend.items():
            lambda_by_nq[(device, kind)].update(fit_runs(backend, kind, stamps))
print(f"{sum(map(len, lambda_by_nq.values()))} fits from data/results in {time.perf_counter() - t0:.0f} s\n")

colours = plt.get_cmap("Set1")
style = {"direct": dict(marker="o", ls="-", ms=7), "mcm": dict(marker="X", ls="--", ms=8)}
fig, ax = plt.subplots(figsize=(6.5, 5))
for device, kinds in LAMBDA_SERIES.items():
    colour = colours(LAMBDA_COLOURS[device])
    labelled = False
    for kind in kinds:
        fits = lambda_by_nq[(device, kind)]
        segments = []                               # consecutive points on the same backend
        for n in sorted(fits):
            if segments and segments[-1][0] == fits[n]["backend"]:
                segments[-1][1].append(n)
            else:
                segments.append((fits[n]["backend"], [n]))
        previous = None
        for backend, nqs in segments:
            hardware_quantinuum = backend in QUANTINUUM_HARDWARE
            ys = [fits[n]["lambda_eff"] for n in nqs]
            ax.plot(nqs, ys, color=colour, mec="black", marker=style[kind]["marker"], ms=style[kind]["ms"],
                    ls=":" if hardware_quantinuum else style[kind]["ls"], label=None if labelled else device)
            labelled = True
            if previous is not None:
                ax.plot([previous[0], nqs[0]], [previous[1], ys[0]], color=colour, ls=":")
            previous = (nqs[-1], ys[-1])
ax.plot([], [], "o", color="black", label="direct")
ax.plot([], [], "X", color="black", ms=8, label="MCM")
ax.set(yscale="log", xlabel="$N_q$", ylabel=r"$\lambda_{\rm eff}$ per edge",
       xticks=sorted({n for fits in lambda_by_nq.values() for n in fits}))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.32), ncols=3, fontsize=9)
fig.savefig(FIGURES / "lambda_eff_by_nq.pdf", bbox_inches="tight")
plt.show()

nqs_all = sorted({n for fits in lambda_by_nq.values() for n in fits})
for kind in ("direct", "mcm"):
    print(f"lambda_eff per edge, {kind}, window p = {LAMBDA_WINDOWS[kind]}"
          + (" (leading plateau dropped)" if DROP_PLATEAU[kind] else ""))
    print(f"{'':<20}" + "".join(f"{n:>10}" for n in nqs_all))
    for device in LAMBDA_SERIES:
        fits = lambda_by_nq.get((device, kind))
        if fits:
            print(f"{device:<20}" + "".join(f"{fits[n]['lambda_eff']:>10.2e}" if n in fits else f"{'-':>10}"
                                            for n in nqs_all))
    worst = max(((f["rmse"], d, n) for (d, k), fits in lambda_by_nq.items() if k == kind for n, f in fits.items()))
    print(f"largest fit RMSE: {worst[0]:.3f} ({worst[1]}, N_q = {worst[2]})\n")

## 8. Save the constants

$\kappa_r(N_q)$, $\kappa_0$, $\kappa_\rho(N_q)$ and the device $\lambda_{\rm eff}$ (sections 6 and 7) are written to
`data/noise_study/kappa_fits.json`, so they can be used without running this notebook again.

In [ ]:
summary = {
    "model": "LR-QAOA on a 1D Ising chain, depolarizing_error(lambda, 2) after every edge interaction",
    "accumulated_error": "N_edges * p * lambda, N_edges = N_q - 1; lambda and lambda_eff are per edge interaction",
    "delta": DELTA, "depths": DEPTHS, "lambdas": list(map(float, LAMBDAS)), "trajectories": TRAJECTORIES,
    "kappa_r": {str(n): fits_r[n] for n in NQS},
    "kappa_0": {"value": kappa0, "stderr": kappa0_err, "rss": kappa0_rss},
    "kappa_rho": {str(n): fits_rho[n] for n in NQS},
    "device": {"n": DEVICE_N, "sim_depths": DEVICE_SIM_DEPTHS,
               "kappa_r": globals().get("deep_r"), "kappa_rho": globals().get("deep_rho")},
    "lambda_eff": {f"{d}|{c}|{o}": f for (d, c, o), f in globals().get("lambda_eff", {}).items()},
    "lambda_eff_by_nq": {
        "windows": LAMBDA_WINDOWS, "drop_plateau": DROP_PLATEAU, "runs": {d: dict(k) for d, k in LAMBDA_SERIES.items()},
        "fits": {f"{device}|{kind}": {str(n): f for n, f in fits.items()}
                 for (device, kind), fits in globals().get("lambda_by_nq", {}).items()}},
}
(DATA / "kappa_fits.json").write_text(json.dumps(summary, indent=1))
print("wrote", (DATA / "kappa_fits.json").relative_to(ROOT))